# Classificação de variedades de feijão seco (Dry Bean)

Este notebook percorre as etapas 2 a 9: descrição, limpeza de linhas, transformação de atributos, partições, comparação de modelos, avaliação e predição. A base é obtida da UCI pela função do projeto.

Fonte: [UCI Dry Bean](https://archive.ics.uci.edu/dataset/602/dry+bean), DOI 10.24432/C50S4B, licença CC BY 4.0.

In [ ]:
from src.pipeline import load_data, describe_data, prepare_rows, make_splits, build_models, TARGET, RANDOM_STATE
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import pandas as pd
import matplotlib.pyplot as plt
import json, joblib
from pathlib import Path

## 1. Carregamento e estatísticas descritivas

In [ ]:
df_original = load_data()
resumo = describe_data(df_original)
print(json.dumps(resumo, indent=2, ensure_ascii=False))
display(df_original.head())
display(df_original.drop(columns=[TARGET]).describe().T)

A remoção de duplicatas exatas é a transformação aplicada às linhas. A padronização dos atributos será aprendida dentro de cada pipeline, somente com o subconjunto de treino.

In [ ]:
df, duplicatas_removidas = prepare_rows(df_original)
print(f'Duplicatas removidas: {duplicatas_removidas}')
print('Linhas antes/depois:', len(df_original), len(df))
X_train, X_val, X_test, y_train, y_val, y_test = make_splits(df)
print({ 'treino': len(X_train), 'validação': len(X_val), 'teste': len(X_test) })
pd.DataFrame({
    'treino': y_train.value_counts(normalize=True),
    'validação': y_val.value_counts(normalize=True),
    'teste': y_test.value_counts(normalize=True)
}).sort_index()

## 2. Comparação na validação

Selecionamos o modelo pelo F1 macro na validação, adequado para resumir o desempenho entre as classes. O teste fica reservado para a avaliação final.

In [ ]:
candidatos = build_models()
resultados = []
for nome, modelo in candidatos.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_val)
    resultados.append({'modelo': nome, 'acurácia_validação': accuracy_score(y_val, pred), 'F1_macro_validação': f1_score(y_val, pred, average='macro')})
comparacao = pd.DataFrame(resultados).sort_values('F1_macro_validação', ascending=False)
display(comparacao)
melhor_nome = comparacao.iloc[0]['modelo']
melhor_nome

## 3. Ajuste final e avaliação no teste

In [ ]:
X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])
modelo_final = clone(candidatos[melhor_nome]).fit(X_trainval, y_trainval)
pred_teste = modelo_final.predict(X_test)
labels = sorted(df[TARGET].astype(str).unique())
print(f'Acurácia: {accuracy_score(y_test, pred_teste):.4f}')
print(f'F1 macro: {f1_score(y_test, pred_teste, average="macro"):.4f}')
print(classification_report(y_test, pred_teste, labels=labels, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, pred_teste, labels=labels, display_labels=labels, xticks_rotation=45, cmap='Blues')
plt.tight_layout()
plt.show()

## 4. Predição demonstrativa e persistência

O exemplo abaixo usa uma observação reservada do conjunto de teste, portanto também podemos conferir a classe verdadeira.

In [ ]:
exemplo = X_test.iloc[[0]]
classe_prevista = modelo_final.predict(exemplo)[0]
prob = modelo_final.predict_proba(exemplo)[0]
confianca = prob[list(modelo_final.classes_).index(classe_prevista)]
print('Classe verdadeira:', y_test.iloc[0])
print('Classe prevista:', classe_prevista)
print(f'Confiança da classe prevista: {confianca:.3f}')
Path('models').mkdir(exist_ok=True)
joblib.dump(modelo_final, 'models/dry_bean_pipeline.joblib')

As métricas serão materializadas em `outputs/` ao executar `python -m src.pipeline`. O README documenta a reprodução, as decisões metodológicas e a estrutura indicada para publicar o código no GitHub.